In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/facknews_data.zip'

extract_path = '/content/facknews_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)



In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import torch
from transformers import BertTokenizer,BertForSequenceClassification,Trainer, TrainingArguments,EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [4]:
data_T=pd.read_csv("/content/facknews_data/facknews_data/True.csv")
data_F=pd.read_csv("/content/facknews_data/facknews_data/Fake.csv")


data_F['label']=0
data_T['label']=1

data = pd.concat([data_F, data_T]).drop(['date','subject'],axis=1)


In [5]:
print(data.info())
data.head(3)

<class 'pandas.core.frame.DataFrame'>
Index: 44898 entries, 0 to 21416
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   44898 non-null  object
 1   text    44898 non-null  object
 2   label   44898 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.4+ MB
None


,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",0


In [6]:
data['content']=(data['title'] + " " + data['text']).astype(str)
x = data['content']
y=data['label'].tolist()

In [7]:
nltk.download('stopwords')
stopwords=set(stopwords.words('english'))
def cleen_text_and_steming(text):
  porter=PorterStemmer()
  text = text.lower()
  text = re.sub(r'\S+@\S+', '', text)
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'http\S+|www\S+', '', text)
  text = re.sub(r'<.*?>', '', text).split()
  text = [w for w in text if w not in stopwords]
  text = [porter.stem(w) for w in text ]
  text = ' '.join(text)
  return text


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
data['content']=data['content'].apply(lambda x: cleen_text_and_steming(x))

In [9]:
train_text,test_text,train_label,test_label=train_test_split(
    x,
    y,
    test_size=0.1,
    random_state=42
)
train_text,val_text,train_label,val_label=train_test_split(
    train_text,
    train_label,
    test_size=0.1,
    random_state=42
)

In [10]:
print(f"train_text : {train_text.shape[0]}")
print(f"test_text : {test_text.shape[0]}")
print(f"val_text : {val_text.shape[0]}")

train_text : 36367
test_text : 4490
val_text : 4041


In [11]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
train_encoder=tokenizer(
    train_text.astype(str).tolist(),
    max_length=512,
    truncation=True ,
    padding='max_length'
    )
val_encoder=tokenizer(
    val_text.astype(str).tolist(),
    max_length=512,
    truncation= True,
    padding='max_length'
  )
test_encoder=tokenizer(
    test_text.astype(str).tolist(),
    max_length=512,
    truncation= True,
    padding='max_length'
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [12]:
model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2)
try :
  from peft import get_peft_model,LoraConfig,TaskType
  confing=LoraConfig(
      task_type=TaskType.SEQ_CLS,
      r=8,
      lora_alpha=32,
      lora_dropout=0.1,
      bias='none'
      )
  model=get_peft_model(model,confing)
  print("Lora is successfully")
except ImportError:
  print("peft is not installing on thes is device ")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Lora is successfully


In [13]:
from torch.utils.data import Dataset
class DataSet(Dataset):
  def __init__(self,encoder,labels):
    self.encoder=encoder
    self.labels=labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self,idx):
    item={key : torch.tensor(val[idx]) for key ,val in self.encoder.items()}
    item['labels']=torch.tensor(self.labels[idx])
    return item


train_dataset=DataSet(train_encoder,train_label)
val_dataset=DataSet(val_encoder,val_label)
test_dataset=DataSet(test_encoder,test_label)


In [14]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [15]:
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

/tmp/ipython-input-2067855592.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.313600
1000,0.044700
1500,0.018200
2000,0.009200
2500,0.004400
3000,0.005400
3500,0.005800
4000,0.004900
4500,0.001200
5000,0.002500


TrainOutput(global_step=6819, training_loss=0.031238206064710696, metrics={'train_runtime': 8041.3181, 'train_samples_per_second': 13.568, 'train_steps_per_second': 0.848, 'total_flos': 2.880503692056576e+16, 'train_loss': 0.031238206064710696, 'epoch': 3.0})

In [16]:
test_metrics = trainer.evaluate(test_dataset)
print("Test set performance:", test_metrics)

Test set performance: {'eval_loss': 0.006521919742226601, 'eval_accuracy': 0.9984409799554566, 'eval_f1': 0.9983400521697889, 'eval_runtime': 141.1787, 'eval_samples_per_second': 31.804, 'eval_steps_per_second': 1.99, 'epoch': 3.0}


In [17]:
save_path = "./fake_news_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


('./fake_news_model/tokenizer_config.json',
 './fake_news_model/special_tokens_map.json',
 './fake_news_model/vocab.txt',
 './fake_news_model/added_tokens.json')